| 0.0 Data Collection | [1.0 Preprocessing](01_data_preprocessing.ipynb) | [2.0 Spatial Autocorrelation](02_spatial_autocorrelation.ipynb) | [3.0 MGWR](03_mgwr_analysis.ipynb) | [4.0 ML Classification](04_ml_classification.ipynb) |

# 0.0 | Data Collection

This notebook downloads all required datasets for the UK Road Traffic Accident spatial analysis:

| Dataset | Source | Format | Approx. Size |
|---------|--------|--------|-------------|
| STATS19 Collisions (2021-2024) | DfT data.gov.uk | CSV | ~20 MB/year |
| MSOA 2021 Boundaries | ONS Open Geography | GeoPackage | ~100 MB |
| LSOA → MSOA Lookup | ONS Open Geography | CSV | <1 MB |
| IMD 2019 Deprivation | MHCLG | CSV | ~10 MB |
| OS Open Roads | Ordnance Survey | GeoPackage | ~1 GB |
| Population Estimates | ONS (Nomis) | CSV | <1 MB |

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.config import ensure_dirs, RAW_DIR
ensure_dirs()
print(f"Raw data directory: {RAW_DIR}")

## 0.1 | STATS19 Collision Data (2021-2024)

The STATS19 dataset is the UK's official record of road traffic collisions reported to the police.  
Each year's CSV contains ~100,000+ collision records with coordinates, severity, road and weather conditions.

**Source**: [Department for Transport](https://www.data.gov.uk/dataset/road-accidents-safety-data)

In [ ]:
from src.scraper.stats19 import download_stats19

stats19_files = download_stats19()
print(f"Downloaded {len(stats19_files)} STATS19 files:")
for f in stats19_files:
    print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")

A quick preview of the 2024 collision data to understand the structure:

In [ ]:
import pandas as pd

df_preview = pd.read_csv(stats19_files[-1], nrows=5)
print(f"Columns ({len(df_preview.columns)}): {list(df_preview.columns)}")
df_preview.head()

## 0.2 | MSOA Boundaries & LSOA-MSOA Lookup

Middle Layer Super Output Areas (MSOAs) are the spatial unit for this analysis.  
There are approximately **6,791 MSOAs** in England, each containing ~7,200 residents.

The LSOA→MSOA lookup table is needed to aggregate IMD 2019 data (published at LSOA level) up to MSOA level.

**Source**: [ONS Open Geography Portal](https://geoportal.statistics.gov.uk/)

In [ ]:
from src.scraper.boundaries import download_msoa_boundaries, download_lsoa_msoa_lookup

msoa_path = download_msoa_boundaries()
lookup_path = download_lsoa_msoa_lookup()
print(f"MSOA boundaries: {msoa_path}")
print(f"LSOA-MSOA lookup: {lookup_path}")

In [ ]:
import geopandas as gpd

msoa_gdf = gpd.read_file(msoa_path)
print(f"MSOA boundaries shape: {msoa_gdf.shape}")
print(f"CRS: {msoa_gdf.crs}")
print(f"Columns: {list(msoa_gdf.columns)}")
msoa_gdf.head(3)

In [ ]:
lookup_df = pd.read_csv(lookup_path)
print(f"Lookup table shape: {lookup_df.shape}")
print(f"Columns: {list(lookup_df.columns)}")
lookup_df.head(3)

## 0.3 | IMD 2019 Deprivation Scores

The Index of Multiple Deprivation (IMD) 2019 measures relative deprivation across seven domains  
(income, employment, education, health, crime, housing, and living environment).  
Higher scores indicate greater deprivation.

**Source**: [MHCLG](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019)

In [ ]:
from src.scraper.imd import download_imd

imd_path = download_imd()
print(f"IMD data: {imd_path} ({imd_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
imd_df = pd.read_csv(imd_path, nrows=5)
print(f"IMD columns ({len(imd_df.columns)}):")
for col in imd_df.columns:
    print(f"  {col}")
imd_df.head()

## 0.4 | OS Open Roads Network

The Ordnance Survey Open Roads dataset provides the national road network geometry,  
used to compute road length and junction density per MSOA.

**Source**: [OS Data Hub](https://osdatahub.os.uk/downloads/open/OpenRoads)

**Warning**: This is a large download (~1 GB). It may take several minutes.

In [ ]:
from src.scraper.os_roads import download_os_roads

roads_path = download_os_roads()
print(f"OS Roads: {roads_path}")

## 0.5 | Population Estimates

Mid-year population estimates at MSOA level are used to normalise accident counts into rates  
(accidents per 10,000 population).

**Source**: [ONS via Nomis](https://www.nomisweb.co.uk/)

In [ ]:
from src.scraper.population import download_population

pop_path = download_population()
print(f"Population data: {pop_path}")

In [ ]:
pop_df = pd.read_csv(pop_path)
print(f"Population data shape: {pop_df.shape}")
print(f"\nSummary statistics:")
pop_df.describe()

## 0.6 | Verification

Confirm all datasets have been downloaded successfully.

In [ ]:
import os

print("=" * 60)
print("DATA COLLECTION SUMMARY")
print("=" * 60)
total_size = 0
for f in sorted(RAW_DIR.rglob('*')):
    if f.is_file() and f.name != '.gitkeep':
        size = f.stat().st_size / 1e6
        total_size += size
        print(f"  {f.relative_to(RAW_DIR)!s:45s} {size:8.1f} MB")
print(f"{'':45s} {'─' * 11}")
print(f"  {'Total':45s} {total_size:8.1f} MB")
print("=" * 60)
print("All datasets downloaded. Proceed to notebook 01_data_preprocessing.")

---
| 0.0 Data Collection | [1.0 Preprocessing](01_data_preprocessing.ipynb) | [2.0 Spatial Autocorrelation](02_spatial_autocorrelation.ipynb) | [3.0 MGWR](03_mgwr_analysis.ipynb) | [4.0 ML Classification](04_ml_classification.ipynb) |